In [1]:
import pandas as pd
import numpy as np

In [2]:
archivo_entrada = "avisos_aemet_geolocalizados_con_id.csv"

df = pd.read_csv(archivo_entrada)

df.head()

,año,mes,mes_num,estacion,event,severity,riesgo_numerico,duracion_horas,area_desc,description,...,hora_fin,minuto_fin,trimestre,valor_numerico,unidad_medida,provincia_limpia,latitud,longitud,provincia_real,id_aviso
0,2021,Enero,1,Invierno,Aviso de temperaturas mínimas de nivel naranja,Severe,2,8.999722,Pirineo oscense,Temperatura mínima: -10 ºC.,...,7,59,1,-10.0,ºC,Pirineo oscense,NaN,NaN,Desconocido,1
1,2021,Enero,1,Invierno,Aviso de temperaturas mínimas de nivel naranja,Severe,2,8.999722,Pirineo de Girona,Temperatura mínima: -10 ºC.,...,7,59,1,-10.0,ºC,Pirineo de Girona,42.400390,1.884586,Catalunya,2
2,2021,Enero,1,Invierno,Aviso de temperaturas mínimas de nivel naranja,Severe,2,8.999722,Valle de Arán,Temperatura mínima: -10 ºC.,...,7,59,1,-10.0,ºC,Valle de Arán,42.725606,0.850352,Catalunya,3
3,2021,Enero,1,Invierno,Aviso de temperaturas mínimas de nivel naranja,Severe,2,8.999722,Pirineo de Lleida,Temperatura mínima: -10 ºC.,...,7,59,1,-10.0,ºC,Pirineo de Lleida,42.400390,1.884586,Catalunya,4
4,2020,Diciembre,12,Invierno,Aviso de nevadas de nivel naranja,Severe,2,26.999722,Cordillera y Picos de Europa,Acumulación de nieve en 24 horas: 20 cm. Esta ...,...,22,59,4,20.0,cm,Cordillera y Picos de Europa,NaN,NaN,Desconocido,5


In [3]:
print("Filas:", len(df))
print("Columnas:", df.shape[1])

df.columns

Filas: 21395
Columnas: 27


Index(['año', 'mes', 'mes_num', 'estacion', 'event', 'severity',
       'riesgo_numerico', 'duracion_horas', 'area_desc', 'description',
       'fecha_inicio', 'fecha_fin', 'fenomeno', 'fecha_inicio_dia',
       'hora_inicio', 'minuto_inicio', 'fecha_fin_dia', 'hora_fin',
       'minuto_fin', 'trimestre', 'valor_numerico', 'unidad_medida',
       'provincia_limpia', 'latitud', 'longitud', 'provincia_real',
       'id_aviso'],
      dtype='object')

In [4]:
print("Latitudes vacías:", df["latitud"].isna().sum())
print("Longitudes vacías:", df["longitud"].isna().sum())

zonas_sin_coords = (
    df[df["latitud"].isna() | df["longitud"].isna()]
    ["area_desc"]
    .value_counts()
    .reset_index()
)

zonas_sin_coords.columns = ["area_desc", "num_avisos"]

zonas_sin_coords.head(30)

Latitudes vacías: 18409
Longitudes vacías: 18409


,area_desc,num_avisos
0,Costa - Noroeste de A Coruña,541
1,Costa - Oeste de A Coruña,528
2,Costa - Suroeste de A Coruña,419
3,Costa - Rias Baixas,388
4,Costa - Miño de Pontevedra,374
5,Costa - A Mariña,334
6,Pirineo oscense,327
7,Costa - Litoral cántabro,307
8,Costa - Litoral oriental asturiano,264
9,Costa - Litoral occidental asturiano,253


In [5]:
df["latitud_corregida"] = df["latitud"]
df["longitud_corregida"] = df["longitud"]

df["fuente_coordenada"] = np.where(
    df["latitud"].notna() & df["longitud"].notna(),
    "original",
    "sin coordenada"
)

df[["area_desc", "latitud", "longitud", "latitud_corregida", "longitud_corregida", "fuente_coordenada"]].head()

,area_desc,latitud,longitud,latitud_corregida,longitud_corregida,fuente_coordenada
0,Pirineo oscense,NaN,NaN,NaN,NaN,sin coordenada
1,Pirineo de Girona,42.400390,1.884586,42.400390,1.884586,original
2,Valle de Arán,42.725606,0.850352,42.725606,0.850352,original
3,Pirineo de Lleida,42.400390,1.884586,42.400390,1.884586,original
4,Cordillera y Picos de Europa,NaN,NaN,NaN,NaN,sin coordenada


In [6]:
coords_por_area = (
    df.dropna(subset=["latitud", "longitud"])
      .groupby("area_desc")[["latitud", "longitud"]]
      .mean()
      .reset_index()
)

coords_por_area = coords_por_area.rename(columns={
    "latitud": "latitud_area",
    "longitud": "longitud_area"
})

df = df.merge(coords_por_area, on="area_desc", how="left")

df["latitud_corregida"] = df["latitud_corregida"].fillna(df["latitud_area"])
df["longitud_corregida"] = df["longitud_corregida"].fillna(df["longitud_area"])

df.loc[
    df["latitud"].isna() & df["latitud_area"].notna(),
    "fuente_coordenada"
] = "media_area_desc"

df.loc[
    df["latitud_corregida"].isna() | df["longitud_corregida"].isna(),
    "fuente_coordenada"
] = "sin coordenada"

print("Después del relleno automático:")
print("Latitudes corregidas vacías:", df["latitud_corregida"].isna().sum())
print("Longitudes corregidas vacías:", df["longitud_corregida"].isna().sum())

Después del relleno automático:
Latitudes corregidas vacías: 18409
Longitudes corregidas vacías: 18409


In [7]:
coordenadas_manual = {
    # Madrid
    "Sierra de Madrid": (40.7796, -3.8196),
    "Metropolitana y Henares": (40.4168, -3.7038),
    "Sur, Vegas y Oeste": (40.3083, -3.7329),

    # Galicia - Costa
    "Costa - Noroeste de A Coruña": (43.3650, -8.2100),
    "Costa - Oeste de A Coruña": (43.0500, -9.0000),
    "Costa - Suroeste de A Coruña": (42.6500, -9.0300),
    "Costa - Miño de Pontevedra": (42.1000, -8.6500),
    "Costa - Rias Baixas": (42.4920, -8.8660),
    "Costa - A Mariña": (43.6700, -7.4100),

    # Asturias / Cantabria
    "Costa - Litoral cántabro": (43.4623, -3.8099),
    "Costa - Litoral oriental asturiano": (43.4300, -4.7500),
    "Costa - Litoral occidental asturiano": (43.5500, -6.5300),

    # Baleares
    "Costa - Menorca": (39.9496, 4.1104),
    "Menorca": (39.9496, 4.1104),
    "Norte y nordeste de Mallorca": (39.8500, 3.0500),
    "Costa - Norte y nordeste de Mallorca": (39.8500, 3.0500),
    "Sur de Mallorca": (39.3500, 2.9500),
    "Interior de Mallorca": (39.6000, 2.9500),
    "Sierra Tramontana": (39.7500, 2.6500),
    "Levante mallorquín": (39.6000, 3.3000),
    "Ibiza y Formentera": (38.9800, 1.4300),

    # Cataluña
    "Prelitoral sur de Tarragona": (41.1500, 0.8500),
    "Costa - Litoral sur de Tarragona": (40.8000, 0.7000),
    "Litoral sur de Tarragona": (40.8000, 0.7000),
    "Depresión central de Barcelona": (41.5000, 1.9000),
    "Prepirineo de Barcelona": (42.0000, 1.9000),
    "Depresión central de Lleida": (41.6176, 0.6200),

    # Aragón
    "Pirineo oscense": (42.6280, -0.3210),
    "Sur de Huesca": (41.9100, -0.2000),
    "Centro de Huesca": (42.1362, -0.4087),
    "Ribera del Ebro de Zaragoza": (41.6488, -0.8891),
    "Ibérica zaragozana": (41.3530, -1.6430),
    "Cinco Villas de Zaragoza": (42.1260, -1.1370),
    "Bajo Aragón de Teruel": (40.8320, -0.2420),

    # Comunidad Valenciana
    "Litoral sur de Valencia": (39.0000, -0.2000),
    "Litoral norte de Valencia": (39.6500, -0.2500),
    "Interior sur de Valencia": (39.0000, -0.7500),
    "Interior norte de Valencia": (39.7000, -0.8000),
    "Interior norte de Castellón": (40.3500, -0.1500),
    "Interior sur de Castellón": (39.9500, -0.3000),
    "Litoral norte de Castellón": (40.0000, 0.0300),
    "Litoral sur de Castellón": (39.8000, -0.0500),
    "Litoral norte de Alicante": (38.6500, -0.0500),
    "Litoral sur de Alicante": (38.2500, -0.6000),

    # Murcia
    "Vega del Segura": (38.0000, -1.0000),

    # Andalucía
    "Campiña cordobesa": (37.8847, -4.7792),
    "Campiña sevillana": (37.3891, -5.9845),
    "Costa - Costa granadina": (36.7300, -3.6900),
    "Costa - Levante almeriense": (37.1500, -1.8500),
    "Costa - Poniente y Almería Capital": (36.8402, -2.4679),
    "Costa - Estrecho": (36.0500, -5.6000),
    "Litoral de Huelva": (37.2500, -6.9500),
    "Cuenca del Genil": (37.2500, -3.7000),
    "Valle del Guadalquivir de Jaén": (37.7796, -3.7849),
    "Morena y Condado": (38.0000, -3.8000),
    "Valle del Almanzora y Los Vélez": (37.3500, -2.3000),
    "Andévalo y Condado": (37.5000, -6.7000),

    # Castilla-La Mancha
    "La Mancha albaceteña": (39.0000, -1.8500),
    "Hellín y Almansa": (38.7000, -1.6000),
    "Valle del Tajo": (39.9600, -4.8300),

    # Extremadura
    "La Siberia extremeña": (39.1000, -5.2000),
    "Villuercas y Montánchez": (39.3000, -5.6000),
    "Vegas del Guadiana": (38.8800, -6.9700),

    # Castilla y León
    "Sur de Ávila": (40.2500, -4.7000),

    # Navarra / La Rioja
    "Pirineo navarro": (42.9000, -1.1000),
    "Ribera del Ebro de Navarra": (42.0600, -1.6050),
    "Centro de Navarra": (42.7000, -1.6500),
    "Ribera del Ebro de La Rioja": (42.4500, -2.4500),

    # Canarias
    "La Gomera": (28.1000, -17.2300),
    "Este, sur y oeste de Gran Canaria": (27.9200, -15.5500),
}

In [8]:
for area, (lat, lon) in coordenadas_manual.items():
    condicion = (
        df["area_desc"].eq(area)
        & (
            df["latitud_corregida"].isna()
            | df["longitud_corregida"].isna()
        )
    )

    df.loc[condicion, "latitud_corregida"] = lat
    df.loc[condicion, "longitud_corregida"] = lon
    df.loc[condicion, "fuente_coordenada"] = "manual"

print("Diccionario manual aplicado.")

Diccionario manual aplicado.


In [9]:
faltantes = df[
    df["latitud_corregida"].isna()
    | df["longitud_corregida"].isna()
]

print("Después de corregir:")
print("Latitudes corregidas vacías:", df["latitud_corregida"].isna().sum())
print("Longitudes corregidas vacías:", df["longitud_corregida"].isna().sum())
print("Zonas que siguen sin coordenadas:", faltantes["area_desc"].nunique())

top_faltantes = (
    faltantes["area_desc"]
    .value_counts()
    .reset_index()
)

top_faltantes.columns = ["area_desc", "num_avisos"]

top_faltantes.head(50)

Después de corregir:
Latitudes corregidas vacías: 6615
Longitudes corregidas vacías: 6615
Zonas que siguen sin coordenadas: 131


,area_desc,num_avisos
0,Gúdar y Maestrazgo,102
1,Prelitoral de Barcelona,102
2,Campiña gaditana,101
3,Norte de Cáceres,101
4,Cumbres de Gran Canaria,101
5,Tajo y Alagón,100
6,Sol y Guadalhorce,100
7,Albarracín y Jiloca,100
8,Costa - Sierra Tramontana,97
9,Alcaraz y Segura,97


In [10]:
df[df["area_desc"].str.contains("Madrid", case=False, na=False)][
    ["area_desc", "fenomeno", "fecha_inicio", "latitud_corregida", "longitud_corregida", "fuente_coordenada"]
].head(20)

,area_desc,fenomeno,fecha_inicio,latitud_corregida,longitud_corregida,fuente_coordenada
250,Sierra de Madrid,Nevadas,2021-01-08 17:00:00+00:00,40.7796,-3.8196,manual
291,Sierra de Madrid,Nevadas,2021-01-08 23:00:00+00:00,40.7796,-3.8196,manual
331,Sierra de Madrid,Nevadas,2021-01-08 11:00:00+00:00,40.7796,-3.8196,manual
403,Sierra de Madrid,Nevadas,2021-01-08 11:00:00+00:00,40.7796,-3.8196,manual
596,Sierra de Madrid,Temperaturas mínimas,2021-01-11 23:00:00+00:00,40.7796,-3.8196,manual
874,Sierra de Madrid,Vientos,2021-01-22 03:00:00+00:00,40.7796,-3.8196,manual
1813,Sierra de Madrid,Temperaturas máximas,2021-07-11 11:00:00+00:00,40.7796,-3.8196,manual
2188,Sierra de Madrid,Temperaturas máximas,2021-08-13 12:00:00+00:00,40.7796,-3.8196,manual
2248,Sierra de Madrid,Temperaturas máximas,2021-08-14 12:00:00+00:00,40.7796,-3.8196,manual
2497,Sierra de Madrid,Temperaturas máximas,2021-08-14 11:00:00+00:00,40.7796,-3.8196,manual


In [11]:
df["fecha_inicio_dt"] = pd.to_datetime(df["fecha_inicio"], errors="coerce")

filomena_madrid = df[
    (df["fecha_inicio_dt"] >= "2021-01-06")
    & (df["fecha_inicio_dt"] <= "2021-01-11")
    & (df["area_desc"].str.contains("Madrid", case=False, na=False))
]

filomena_madrid[
    ["area_desc", "fenomeno", "fecha_inicio", "fecha_fin", "latitud_corregida", "longitud_corregida", "fuente_coordenada"]
].head(30)

,area_desc,fenomeno,fecha_inicio,fecha_fin,latitud_corregida,longitud_corregida,fuente_coordenada
250,Sierra de Madrid,Nevadas,2021-01-08 17:00:00+00:00,2021-01-08 22:59:59+00:00,40.7796,-3.8196,manual
291,Sierra de Madrid,Nevadas,2021-01-08 23:00:00+00:00,2021-01-09 22:59:59+00:00,40.7796,-3.8196,manual
331,Sierra de Madrid,Nevadas,2021-01-08 11:00:00+00:00,2021-01-08 22:59:59+00:00,40.7796,-3.8196,manual
403,Sierra de Madrid,Nevadas,2021-01-08 11:00:00+00:00,2021-01-09 22:59:59+00:00,40.7796,-3.8196,manual


In [12]:
df["fuente_coordenada"].value_counts()

fuente_coordenada
manual            11794
sin coordenada     6615
original           2986
Name: count, dtype: int64

In [13]:
# Crear provincia corregida a partir de provincia_real
df["provincia_corregida"] = df["provincia_real"]

# Si está vacía o como Desconocido, la dejamos como pendiente
df.loc[
    df["provincia_corregida"].isna() | (df["provincia_corregida"] == "Desconocido"),
    "provincia_corregida"
] = np.nan

In [14]:
diccionario_areas = {
    # Madrid
    "Sierra de Madrid": (40.7796, -3.8196, "Madrid"),
    "Metropolitana y Henares": (40.4168, -3.7038, "Madrid"),
    "Sur, Vegas y Oeste": (40.3083, -3.7329, "Madrid"),

    # Galicia
    "Costa - Noroeste de A Coruña": (43.3650, -8.2100, "A Coruña"),
    "Costa - Oeste de A Coruña": (43.0500, -9.0000, "A Coruña"),
    "Costa - Suroeste de A Coruña": (42.6500, -9.0300, "A Coruña"),
    "Costa - Miño de Pontevedra": (42.1000, -8.6500, "Pontevedra"),
    "Costa - Rias Baixas": (42.4920, -8.8660, "Pontevedra"),
    "Costa - A Mariña": (43.6700, -7.4100, "Lugo"),

    # Asturias / Cantabria
    "Costa - Litoral cántabro": (43.4623, -3.8099, "Cantabria"),
    "Costa - Litoral oriental asturiano": (43.4300, -4.7500, "Asturias"),
    "Costa - Litoral occidental asturiano": (43.5500, -6.5300, "Asturias"),
    "Liébana": (43.1500, -4.6500, "Cantabria"),

    # Baleares
    "Costa - Menorca": (39.9496, 4.1104, "Illes Balears"),
    "Menorca": (39.9496, 4.1104, "Illes Balears"),
    "Norte y nordeste de Mallorca": (39.8500, 3.0500, "Illes Balears"),
    "Costa - Norte y nordeste de Mallorca": (39.8500, 3.0500, "Illes Balears"),
    "Sur de Mallorca": (39.3500, 2.9500, "Illes Balears"),
    "Costa - Sur de Mallorca": (39.3500, 2.9500, "Illes Balears"),
    "Interior de Mallorca": (39.6000, 2.9500, "Illes Balears"),
    "Sierra Tramontana": (39.7500, 2.6500, "Illes Balears"),
    "Costa - Sierra Tramontana": (39.7500, 2.6500, "Illes Balears"),
    "Levante mallorquín": (39.6000, 3.3000, "Illes Balears"),
    "Costa - Levante mallorquín": (39.6000, 3.3000, "Illes Balears"),
    "Ibiza y Formentera": (38.9800, 1.4300, "Illes Balears"),
    "Costa - Ibiza y Formentera": (38.9800, 1.4300, "Illes Balears"),

    # Cataluña
    "Prelitoral de Barcelona": (41.6000, 1.9000, "Barcelona"),
    "Litoral de Barcelona": (41.3900, 2.1700, "Barcelona"),
    "Depresión central de Barcelona": (41.5000, 1.9000, "Barcelona"),
    "Prepirineo de Barcelona": (42.0000, 1.9000, "Barcelona"),
    "Prelitoral de Girona": (41.9500, 2.6500, "Girona"),
    "Prelitoral sur de Tarragona": (41.1500, 0.8500, "Tarragona"),
    "Prelitoral norte de Tarragona": (41.3000, 1.2500, "Tarragona"),
    "Costa - Litoral sur de Tarragona": (40.8000, 0.7000, "Tarragona"),
    "Litoral sur de Tarragona": (40.8000, 0.7000, "Tarragona"),
    "Litoral norte de Tarragona": (41.1500, 1.3000, "Tarragona"),
    "Depresión central de Tarragona": (41.2000, 1.1000, "Tarragona"),
    "Depresión central de Lleida": (41.6176, 0.6200, "Lleida"),

    # Aragón
    "Pirineo oscense": (42.6280, -0.3210, "Huesca"),
    "Sur de Huesca": (41.9100, -0.2000, "Huesca"),
    "Centro de Huesca": (42.1362, -0.4087, "Huesca"),
    "Ribera del Ebro de Zaragoza": (41.6488, -0.8891, "Zaragoza"),
    "Ibérica zaragozana": (41.3530, -1.6430, "Zaragoza"),
    "Cinco Villas de Zaragoza": (42.1260, -1.1370, "Zaragoza"),
    "Bajo Aragón de Teruel": (40.8320, -0.2420, "Teruel"),
    "Gúdar y Maestrazgo": (40.3500, -0.6500, "Teruel"),
    "Albarracín y Jiloca": (40.6500, -1.3500, "Teruel"),

    # Comunidad Valenciana
    "Litoral sur de Valencia": (39.0000, -0.2000, "Valencia"),
    "Litoral norte de Valencia": (39.6500, -0.2500, "Valencia"),
    "Interior sur de Valencia": (39.0000, -0.7500, "Valencia"),
    "Interior norte de Valencia": (39.7000, -0.8000, "Valencia"),
    "Interior norte de Castellón": (40.3500, -0.1500, "Castellón"),
    "Interior sur de Castellón": (39.9500, -0.3000, "Castellón"),
    "Litoral norte de Castellón": (40.0000, 0.0300, "Castellón"),
    "Litoral sur de Castellón": (39.8000, -0.0500, "Castellón"),
    "Litoral norte de Alicante": (38.6500, -0.0500, "Alicante"),
    "Litoral sur de Alicante": (38.2500, -0.6000, "Alicante"),

    # Murcia
    "Vega del Segura": (38.0000, -1.0000, "Murcia"),
    "Valle del Guadalentín, Lorca y Águilas": (37.6500, -1.7000, "Murcia"),
    "Altiplano de Murcia": (38.6000, -1.3000, "Murcia"),
    "Campo de Cartagena y Mazarrón": (37.6000, -1.0500, "Murcia"),

    # Andalucía
    "Campiña cordobesa": (37.8847, -4.7792, "Córdoba"),
    "Sierra y Pedroches": (38.3000, -4.7000, "Córdoba"),
    "Subbética cordobesa": (37.4000, -4.3000, "Córdoba"),
    "Campiña sevillana": (37.3891, -5.9845, "Sevilla"),
    "Sierra norte de Sevilla": (37.9000, -5.8000, "Sevilla"),
    "Campiña gaditana": (36.6500, -5.9000, "Cádiz"),
    "Costa - Litoral gaditano": (36.5000, -6.2000, "Cádiz"),
    "Grazalema": (36.7600, -5.3700, "Cádiz"),
    "Estrecho": (36.0500, -5.6000, "Cádiz"),
    "Sol y Guadalhorce": (36.7200, -4.4200, "Málaga"),
    "Costa - Costa granadina": (36.7300, -3.6900, "Granada"),
    "Cuenca del Genil": (37.2500, -3.7000, "Granada"),
    "Costa - Levante almeriense": (37.1500, -1.8500, "Almería"),
    "Costa - Poniente y Almería Capital": (36.8402, -2.4679, "Almería"),
    "Valle del Almanzora y Los Vélez": (37.3500, -2.3000, "Almería"),
    "Litoral de Huelva": (37.2500, -6.9500, "Huelva"),
    "Andévalo y Condado": (37.5000, -6.7000, "Huelva"),
    "Aracena": (37.9000, -6.5500, "Huelva"),
    "Valle del Guadalquivir de Jaén": (37.7796, -3.7849, "Jaén"),
    "Morena y Condado": (38.0000, -3.8000, "Jaén"),

    # Castilla-La Mancha
    "La Mancha albaceteña": (39.0000, -1.8500, "Albacete"),
    "Hellín y Almansa": (38.7000, -1.6000, "Albacete"),
    "Alcaraz y Segura": (38.5000, -2.2500, "Albacete"),
    "La Mancha toledana": (39.4500, -3.4000, "Toledo"),
    "Sierra de San Vicente": (40.1000, -4.7000, "Toledo"),
    "La Mancha conquense": (39.6000, -2.2000, "Cuenca"),
    "Alcarria de Guadalajara": (40.7000, -2.8000, "Guadalajara"),
    "La Mancha de Ciudad Real": (39.0000, -3.9000, "Ciudad Real"),
    "Valle del Tajo": (39.9600, -4.8300, "Toledo"),

    # Extremadura
    "Norte de Cáceres": (40.1500, -5.8000, "Cáceres"),
    "Tajo y Alagón": (39.8500, -6.1000, "Cáceres"),
    "Meseta cacereña": (39.4700, -6.3700, "Cáceres"),
    "La Siberia extremeña": (39.1000, -5.2000, "Badajoz"),
    "Villuercas y Montánchez": (39.3000, -5.6000, "Cáceres"),
    "Vegas del Guadiana": (38.8800, -6.9700, "Badajoz"),
    "Valle del Guadiana": (38.8800, -6.9700, "Badajoz"),
    "Barros y Serena": (38.7500, -6.2000, "Badajoz"),
    "Sur de Badajoz": (38.2500, -6.7000, "Badajoz"),

    # Castilla y León
    "Sur de Ávila": (40.2500, -4.7000, "Ávila"),
    "Norte de Burgos": (42.9000, -3.6500, "Burgos"),
    "Cordillera Cantábrica de Burgos": (43.0000, -3.6000, "Burgos"),
    "Cordillera Cantábrica de León": (42.9500, -5.3000, "León"),
    "Cordillera y Picos de Europa": (43.1870, -4.8220, "León"),
    "Meseta de Valladolid": (41.6500, -4.7200, "Valladolid"),

    # Navarra / La Rioja / País Vasco
    "Pirineo navarro": (42.9000, -1.1000, "Navarra"),
    "Ribera del Ebro de Navarra": (42.0600, -1.6050, "Navarra"),
    "Centro de Navarra": (42.7000, -1.6500, "Navarra"),
    "Vertiente cantábrica de Navarra": (43.0500, -1.6500, "Navarra"),
    "Ribera del Ebro de La Rioja": (42.4500, -2.4500, "La Rioja"),
    "Ibérica riojana": (42.2000, -2.6000, "La Rioja"),
    "Cuenca del Nervión": (43.2500, -2.9500, "Bizkaia"),
    "Condado de Treviño": (42.7300, -2.7500, "Burgos"),

    # Canarias
    "La Gomera": (28.1000, -17.2300, "Santa Cruz de Tenerife"),
    "El Hierro": (27.7300, -18.0200, "Santa Cruz de Tenerife"),
    "Este de La Palma": (28.6800, -17.7800, "Santa Cruz de Tenerife"),
    "Este, sur y oeste de Tenerife": (28.2000, -16.6000, "Santa Cruz de Tenerife"),
    "Cumbres de Gran Canaria": (27.9700, -15.5800, "Las Palmas"),
    "Este, sur y oeste de Gran Canaria": (27.9200, -15.5500, "Las Palmas"),
}

In [15]:
for area, (lat, lon, provincia) in diccionario_areas.items():
    condicion_area = df["area_desc"].eq(area)

    # Rellenar coordenadas solo si están vacías
    condicion_coords_vacias = (
        condicion_area
        & (
            df["latitud_corregida"].isna()
            | df["longitud_corregida"].isna()
        )
    )

    df.loc[condicion_coords_vacias, "latitud_corregida"] = lat
    df.loc[condicion_coords_vacias, "longitud_corregida"] = lon
    df.loc[condicion_coords_vacias, "fuente_coordenada"] = "manual"

    # Rellenar provincia si está vacía o desconocida
    condicion_provincia_vacia = (
        condicion_area
        & (
            df["provincia_corregida"].isna()
            | (df["provincia_corregida"] == "Desconocido")
        )
    )

    df.loc[condicion_provincia_vacia, "provincia_corregida"] = provincia

print("Coordenadas y provincias corregidas con diccionario manual.")

Coordenadas y provincias corregidas con diccionario manual.


In [16]:
faltantes_coords = df[
    df["latitud_corregida"].isna()
    | df["longitud_corregida"].isna()
]

faltantes_provincia = df[
    df["provincia_corregida"].isna()
    | (df["provincia_corregida"] == "Desconocido")
]

print("Coordenadas pendientes:")
print("Filas sin coordenadas:", len(faltantes_coords))
print("Zonas sin coordenadas:", faltantes_coords["area_desc"].nunique())

print("\nProvincias pendientes:")
print("Filas sin provincia:", len(faltantes_provincia))
print("Zonas sin provincia:", faltantes_provincia["area_desc"].nunique())

Coordenadas pendientes:
Filas sin coordenadas: 2658
Zonas sin coordenadas: 81

Provincias pendientes:
Filas sin provincia: 2777
Zonas sin provincia: 82


In [17]:
top_faltantes_coords = (
    df[
        df["latitud_corregida"].isna()
        | df["longitud_corregida"].isna()
    ]["area_desc"]
    .value_counts()
    .reset_index()
)

top_faltantes_coords.columns = ["area_desc", "num_avisos"]

top_faltantes_coords

,area_desc,num_avisos
0,Cordillera Cantábrica de Palencia,57
1,Miño de Ourense,56
2,Alcarria conquense,55
3,Meseta de Soria,54
4,Poniente y Almería Capital,54
...,...,...
76,"Costa - Este, sur y oeste de Gran Canaria",12
77,Costa - La Gomera,11
78,Costa - Litoral de Huelva,11
79,Litoral occidental asturiano,6


In [18]:
pd.set_option("display.max_rows", 100)

top_faltantes_coords = (
    df[
        df["latitud_corregida"].isna()
        | df["longitud_corregida"].isna()
    ]["area_desc"]
    .value_counts()
    .reset_index()
)

top_faltantes_coords.columns = ["area_desc", "num_avisos"]

top_faltantes_coords

,area_desc,num_avisos
0,Cordillera Cantábrica de Palencia,57
1,Miño de Ourense,56
2,Alcarria conquense,55
3,Meseta de Soria,54
4,Poniente y Almería Capital,54
5,Sierras de Alcudia y Madrona,54
6,Levante almeriense,54
7,Sistema Central de Segovia,53
8,Área metropolitana de Tenerife,53
9,Rioja alavesa,53


In [19]:
pd.set_option("display.max_rows", 100)

top_faltantes_provincia = (
    df[
        df["provincia_corregida"].isna()
        | (df["provincia_corregida"] == "Desconocido")
    ]["area_desc"]
    .value_counts()
    .reset_index()
)

top_faltantes_provincia.columns = ["area_desc", "num_avisos"]

top_faltantes_provincia

,area_desc,num_avisos
0,Costa - Estrecho,119
1,Cordillera Cantábrica de Palencia,57
2,Miño de Ourense,56
3,Alcarria conquense,55
4,Meseta de Soria,54
5,Poniente y Almería Capital,54
6,Sierras de Alcudia y Madrona,54
7,Levante almeriense,54
8,Área metropolitana de Tenerife,53
9,Sistema Central de Segovia,53


In [20]:
# Añadimos también "Costa - Estrecho", que faltaba en la tanda anterior
diccionario_areas_2["Costa - Estrecho"] = (36.0500, -5.6000, "Cádiz")

# Aplicar diccionario final de coordenadas y provincias
for area, (lat, lon, provincia) in diccionario_areas_2.items():
    condicion_area = df["area_desc"].eq(area)

    # Coordenadas
    condicion_coords_vacias = (
        condicion_area
        & (
            df["latitud_corregida"].isna()
            | df["longitud_corregida"].isna()
        )
    )

    df.loc[condicion_coords_vacias, "latitud_corregida"] = lat
    df.loc[condicion_coords_vacias, "longitud_corregida"] = lon
    df.loc[condicion_coords_vacias, "fuente_coordenada"] = "manual_2"

    # Provincia
    condicion_provincia_vacia = (
        condicion_area
        & (
            df["provincia_corregida"].isna()
            | (df["provincia_corregida"] == "Desconocido")
        )
    )

    df.loc[condicion_provincia_vacia, "provincia_corregida"] = provincia

print("Diccionario final aplicado correctamente.")

NameError: name 'diccionario_areas_2' is not defined

In [21]:
diccionario_areas_2 = {
    # Andalucía
    "Costa - Estrecho": (36.0500, -5.6000, "Cádiz"),
    "Poniente y Almería Capital": (36.8402, -2.4679, "Almería"),
    "Levante almeriense": (37.1500, -1.8500, "Almería"),
    "Litoral gaditano": (36.5000, -6.2000, "Cádiz"),
    "Ronda": (36.7400, -5.1600, "Málaga"),
    "Nacimiento y Campo de Tabernas": (37.0500, -2.4000, "Almería"),
    "Capital y Montes de Jaén": (37.7800, -3.7900, "Jaén"),
    "Sierra sur de Sevilla": (37.0500, -5.4500, "Sevilla"),
    "Antequera": (37.0200, -4.5600, "Málaga"),
    "Axarquía": (36.8000, -4.1000, "Málaga"),
    "Costa - Axarquía": (36.7500, -3.9500, "Málaga"),
    "Costa - Sol y Guadalhorce": (36.7200, -4.4200, "Málaga"),
    "Nevada y Alpujarras": (37.0500, -3.2500, "Granada"),
    "Costa granadina": (36.7300, -3.6900, "Granada"),

    # Castilla y León
    "Cordillera Cantábrica de Palencia": (42.9000, -4.5000, "Palencia"),
    "Meseta de Soria": (41.7600, -2.4700, "Soria"),
    "Sistema Central de Segovia": (41.1000, -3.9000, "Segovia"),
    "Meseta de Burgos": (42.3400, -3.7000, "Burgos"),
    "Meseta de Segovia": (41.2000, -4.0000, "Segovia"),
    "Meseta de Zamora": (41.5000, -5.7500, "Zamora"),
    "Sistema Central de Soria": (41.8500, -2.8000, "Soria"),
    "Meseta de Salamanca": (40.9700, -5.6600, "Salamanca"),
    "Sur de Salamanca": (40.4000, -5.9000, "Salamanca"),
    "Meseta de Palencia": (42.0000, -4.5300, "Palencia"),
    "Sanabria": (42.1000, -6.7000, "Zamora"),
    "Meseta de Ávila": (40.6500, -4.7000, "Ávila"),
    "Sistema Central de Salamanca": (40.4800, -5.9000, "Salamanca"),
    "Meseta de León": (42.6000, -5.5700, "León"),
    "Bierzo de León": (42.5500, -6.6000, "León"),

    # Galicia
    "Miño de Ourense": (42.3500, -7.8700, "Ourense"),
    "Suroeste de A Coruña": (42.6500, -9.0300, "A Coruña"),
    "Interior de Pontevedra": (42.4300, -8.4500, "Pontevedra"),
    "Miño de Pontevedra": (42.1000, -8.6500, "Pontevedra"),
    "Rias Baixas": (42.4920, -8.8660, "Pontevedra"),
    "Oeste de A Coruña": (43.0500, -9.0000, "A Coruña"),
    "A Mariña": (43.6700, -7.4100, "Lugo"),
    "Sur de Ourense": (42.0000, -7.6000, "Ourense"),
    "Noroeste de A Coruña": (43.3650, -8.2100, "A Coruña"),
    "Sur de Lugo": (42.7000, -7.5000, "Lugo"),
    "Noroeste de Ourense": (42.4000, -8.0000, "Ourense"),
    "Interior de A Coruña": (43.0000, -8.5000, "A Coruña"),
    "Montaña de Lugo": (42.9000, -7.2000, "Lugo"),
    "Montaña de Ourense": (42.2500, -7.2000, "Ourense"),
    "Centro de Lugo": (43.0100, -7.5600, "Lugo"),
    "Valdeorras": (42.4200, -6.9800, "Ourense"),

    # Castilla-La Mancha
    "Alcarria conquense": (40.1500, -2.4000, "Cuenca"),
    "Sierras de Alcudia y Madrona": (38.5000, -4.3000, "Ciudad Real"),
    "Montes de Toledo": (39.5000, -4.0000, "Toledo"),
    "Montes del norte y Anchuras": (39.3000, -4.5000, "Ciudad Real"),

    # Cataluña
    "Costa - Litoral norte de Tarragona": (41.1500, 1.3000, "Tarragona"),
    "Costa - Litoral sur de Girona": (41.7500, 2.9000, "Girona"),
    "Litoral sur de Girona": (41.7500, 2.9000, "Girona"),
    "Costa - Litoral de Barcelona": (41.3900, 2.1700, "Barcelona"),

    # Comunidad Valenciana
    "Costa - Litoral norte de Castellón": (40.0000, 0.0300, "Castellón"),
    "Costa - Litoral norte de Alicante": (38.6500, -0.0500, "Alicante"),
    "Costa - Litoral sur de Valencia": (39.0000, -0.2000, "Valencia"),
    "Costa - Litoral sur de Castellón": (39.8000, -0.0500, "Castellón"),
    "Costa - Litoral norte de Valencia": (39.6500, -0.2500, "Valencia"),
    "Costa - Litoral sur de Alicante": (38.2500, -0.6000, "Alicante"),

    # Murcia
    "Costa - Campo de Cartagena y Mazarrón": (37.6000, -1.0500, "Murcia"),
    "Costa - Valle del Guadalentín, Lorca y Águilas": (37.6500, -1.7000, "Murcia"),

    # Canarias
    "Área metropolitana de Tenerife": (28.4600, -16.2500, "Santa Cruz de Tenerife"),
    "Lanzarote": (29.0400, -13.6300, "Las Palmas"),
    "Fuerteventura": (28.3600, -14.0500, "Las Palmas"),
    "Costa - Lanzarote": (29.0400, -13.6300, "Las Palmas"),
    "Costa - Este de La Palma": (28.6800, -17.7800, "Santa Cruz de Tenerife"),
    "Costa - El Hierro": (27.7300, -18.0200, "Santa Cruz de Tenerife"),
    "Costa - Este, sur y oeste de Tenerife": (28.2000, -16.6000, "Santa Cruz de Tenerife"),
    "Costa - Área metropolitana de Tenerife": (28.4600, -16.2500, "Santa Cruz de Tenerife"),
    "Costa - Fuerteventura": (28.3600, -14.0500, "Las Palmas"),
    "Costa - Este, sur y oeste de Gran Canaria": (27.9200, -15.5500, "Las Palmas"),
    "Costa - La Gomera": (28.1000, -17.2300, "Santa Cruz de Tenerife"),

    # Asturias / Cantabria
    "Suroccidental asturiana": (43.2500, -6.5000, "Asturias"),
    "Litoral cántabro": (43.4623, -3.8099, "Cantabria"),
    "Central y Valles Mineros": (43.2500, -5.8000, "Asturias"),
    "Litoral occidental asturiano": (43.5500, -6.5300, "Asturias"),
    "Litoral oriental asturiano": (43.4300, -4.7500, "Asturias"),
    "Cantabria del Ebro": (42.9000, -4.2000, "Cantabria"),

    # País Vasco / La Rioja
    "Rioja alavesa": (42.5500, -2.6000, "Álava"),

    # Melilla
    "Costa - Melilla": (35.2923, -2.9381, "Melilla"),
    "Melilla": (35.2923, -2.9381, "Melilla"),

    # Huelva costa
    "Costa - Litoral de Huelva": (37.2500, -6.9500, "Huelva"),
}

In [22]:
for area, (lat, lon, provincia) in diccionario_areas_2.items():
    condicion_area = df["area_desc"].eq(area)

    condicion_coords_vacias = (
        condicion_area
        & (
            df["latitud_corregida"].isna()
            | df["longitud_corregida"].isna()
        )
    )

    df.loc[condicion_coords_vacias, "latitud_corregida"] = lat
    df.loc[condicion_coords_vacias, "longitud_corregida"] = lon
    df.loc[condicion_coords_vacias, "fuente_coordenada"] = "manual_2"

    condicion_provincia_vacia = (
        condicion_area
        & (
            df["provincia_corregida"].isna()
            | (df["provincia_corregida"] == "Desconocido")
        )
    )

    df.loc[condicion_provincia_vacia, "provincia_corregida"] = provincia

print("Diccionario final aplicado.")

Diccionario final aplicado.


In [23]:
faltantes_coords = df[
    df["latitud_corregida"].isna()
    | df["longitud_corregida"].isna()
]

faltantes_provincia = df[
    df["provincia_corregida"].isna()
    | (df["provincia_corregida"] == "Desconocido")
]

print("Coordenadas pendientes:")
print("Filas sin coordenadas:", len(faltantes_coords))
print("Zonas sin coordenadas:", faltantes_coords["area_desc"].nunique())

print("\nProvincias pendientes:")
print("Filas sin provincia:", len(faltantes_provincia))
print("Zonas sin provincia:", faltantes_provincia["area_desc"].nunique())

top_faltantes_coords = (
    faltantes_coords["area_desc"]
    .value_counts()
    .reset_index()
)

top_faltantes_coords.columns = ["area_desc", "num_avisos"]

top_faltantes_coords.head(100)


Coordenadas pendientes:
Filas sin coordenadas: 0
Zonas sin coordenadas: 0

Provincias pendientes:
Filas sin provincia: 0
Zonas sin provincia: 0


,area_desc,num_avisos


In [25]:
columnas_clave = [
    "id_aviso",
    "fecha_inicio",
    "fecha_fin",
    "fenomeno",
    "severity",
    "area_desc",
    "provincia_corregida",
    "latitud_corregida",
    "longitud_corregida"
]

df[columnas_clave].isna().sum()

id_aviso               0
fecha_inicio           0
fecha_fin              0
fenomeno               0
severity               0
area_desc              0
provincia_corregida    0
latitud_corregida      0
longitud_corregida     0
dtype: int64

In [26]:
coords_fuera_rango = df[
    (df["latitud_corregida"] < 27)
    | (df["latitud_corregida"] > 44)
    | (df["longitud_corregida"] < -19)
    | (df["longitud_corregida"] > 5)
]

print("Filas con coordenadas fuera de rango:", len(coords_fuera_rango))

coords_fuera_rango[
    ["area_desc", "provincia_corregida", "latitud_corregida", "longitud_corregida"]
].drop_duplicates().head(50)

Filas con coordenadas fuera de rango: 0


,area_desc,provincia_corregida,latitud_corregida,longitud_corregida


In [27]:
df["fuente_coordenada"].value_counts()

fuente_coordenada
manual      15751
original     2986
manual_2     2658
Name: count, dtype: int64

In [28]:
df["provincia_corregida"].value_counts().sort_index()

provincia_corregida
A Coruña                  1632
Albacete                   299
Alicante                   214
Almería                    540
Andalucía                  287
Asturias                   578
Badajoz                    471
Barcelona                  429
Bizkaia                     66
Burgos                     262
Canarias                   135
Cantabria                  539
Castellón                  579
Castilla y León            190
Castilla-La Mancha         235
Catalunya                  895
Ceuta                       43
Ciudad Real                162
Comunitat Valenciana       185
Cuenca                     138
Cáceres                    382
Cádiz                      506
Córdoba                    320
Euskadi                    831
Girona                     162
Granada                    257
Guadalajara                 74
Huelva                     313
Huesca                     585
Illes Balears             1775
Jaén                       346
La Rioja           

In [29]:
df["fecha_inicio_dt"] = pd.to_datetime(df["fecha_inicio"], errors="coerce")

filomena_madrid = df[
    (df["fecha_inicio_dt"] >= "2021-01-06")
    & (df["fecha_inicio_dt"] <= "2021-01-11")
    & (df["area_desc"].str.contains("Madrid", case=False, na=False))
]

print("Filas Filomena Madrid:", len(filomena_madrid))

filomena_madrid[
    [
        "area_desc",
        "provincia_corregida",
        "fenomeno",
        "fecha_inicio",
        "fecha_fin",
        "latitud_corregida",
        "longitud_corregida",
        "fuente_coordenada"
    ]
].head(30)

Filas Filomena Madrid: 4


,area_desc,provincia_corregida,fenomeno,fecha_inicio,fecha_fin,latitud_corregida,longitud_corregida,fuente_coordenada
250,Sierra de Madrid,Madrid,Nevadas,2021-01-08 17:00:00+00:00,2021-01-08 22:59:59+00:00,40.7796,-3.8196,manual
291,Sierra de Madrid,Madrid,Nevadas,2021-01-08 23:00:00+00:00,2021-01-09 22:59:59+00:00,40.7796,-3.8196,manual
331,Sierra de Madrid,Madrid,Nevadas,2021-01-08 11:00:00+00:00,2021-01-08 22:59:59+00:00,40.7796,-3.8196,manual
403,Sierra de Madrid,Madrid,Nevadas,2021-01-08 11:00:00+00:00,2021-01-09 22:59:59+00:00,40.7796,-3.8196,manual


In [30]:
# Crear una copia limpia para Tableau
df_tableau = df.copy()

# Columnas que queremos eliminar si existen
columnas_eliminar = [
    "latitud",
    "longitud",
    "provincia_real",
    "latitud_area",
    "longitud_area",
    "fecha_inicio_dt"
]

df_tableau = df_tableau.drop(
    columns=[col for col in columnas_eliminar if col in df_tableau.columns]
)

# Renombrar columnas corregidas para que Tableau las use directamente
df_tableau = df_tableau.rename(columns={
    "latitud_corregida": "latitud",
    "longitud_corregida": "longitud",
    "provincia_corregida": "provincia"
})

# Ver columnas finales
df_tableau.columns.tolist()

['año',
 'mes',
 'mes_num',
 'estacion',
 'event',
 'severity',
 'riesgo_numerico',
 'duracion_horas',
 'area_desc',
 'description',
 'fecha_inicio',
 'fecha_fin',
 'fenomeno',
 'fecha_inicio_dia',
 'hora_inicio',
 'minuto_inicio',
 'fecha_fin_dia',
 'hora_fin',
 'minuto_fin',
 'trimestre',
 'valor_numerico',
 'unidad_medida',
 'provincia_limpia',
 'id_aviso',
 'latitud',
 'longitud',
 'fuente_coordenada',
 'provincia']

In [31]:
df_tableau[[
    "area_desc",
    "provincia",
    "latitud",
    "longitud",
    "fuente_coordenada"
]].head()

,area_desc,provincia,latitud,longitud,fuente_coordenada
0,Pirineo oscense,Huesca,42.628000,-0.321000,manual
1,Pirineo de Girona,Catalunya,42.400390,1.884586,original
2,Valle de Arán,Catalunya,42.725606,0.850352,original
3,Pirineo de Lleida,Catalunya,42.400390,1.884586,original
4,Cordillera y Picos de Europa,León,43.187000,-4.822000,manual


In [32]:
df_tableau[["provincia", "latitud", "longitud"]].isna().sum()

provincia    0
latitud      0
longitud     0
dtype: int64

In [34]:
df_tableau.head(5)

,año,mes,mes_num,estacion,event,severity,riesgo_numerico,duracion_horas,area_desc,description,...,minuto_fin,trimestre,valor_numerico,unidad_medida,provincia_limpia,id_aviso,latitud,longitud,fuente_coordenada,provincia
0,2021,Enero,1,Invierno,Aviso de temperaturas mínimas de nivel naranja,Severe,2,8.999722,Pirineo oscense,Temperatura mínima: -10 ºC.,...,59,1,-10.0,ºC,Pirineo oscense,1,42.628000,-0.321000,manual,Huesca
1,2021,Enero,1,Invierno,Aviso de temperaturas mínimas de nivel naranja,Severe,2,8.999722,Pirineo de Girona,Temperatura mínima: -10 ºC.,...,59,1,-10.0,ºC,Pirineo de Girona,2,42.400390,1.884586,original,Catalunya
2,2021,Enero,1,Invierno,Aviso de temperaturas mínimas de nivel naranja,Severe,2,8.999722,Valle de Arán,Temperatura mínima: -10 ºC.,...,59,1,-10.0,ºC,Valle de Arán,3,42.725606,0.850352,original,Catalunya
3,2021,Enero,1,Invierno,Aviso de temperaturas mínimas de nivel naranja,Severe,2,8.999722,Pirineo de Lleida,Temperatura mínima: -10 ºC.,...,59,1,-10.0,ºC,Pirineo de Lleida,4,42.400390,1.884586,original,Catalunya
4,2020,Diciembre,12,Invierno,Aviso de nevadas de nivel naranja,Severe,2,26.999722,Cordillera y Picos de Europa,Acumulación de nieve en 24 horas: 20 cm. Esta ...,...,59,4,20.0,cm,Cordillera y Picos de Europa,5,43.187000,-4.822000,manual,León


In [35]:
df_tableau.to_csv(
    "avisos_aemet_tableau_limpio.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivo creado: avisos_aemet_tableau_limpio.csv")

Archivo creado: avisos_aemet_tableau_limpio.csv
